# Week 36

In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
from google.colab import drive
import os
drive.mount('/content/drive')

# Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

In [ ]:
!pip install camel-tools konlpy indic-nlp-library

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 105.9 MB/s eta 0:00:00
   ━━━━━

In [ ]:
from camel_tools.tokenizers.word import simple_word_tokenize
from konlpy.tag import Okt
from indicnlp.tokenize import indic_tokenize

okt = Okt()

langForStat = ['ar', 'ko', 'te']

numQuestions = []
totalWordCount = []
distinctWordCount = []
distinctCharCount = []

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

def tokenize_by_lang(text, lang):
    if not isinstance(text, str):
        return []
    if lang == 'ar':
        return simple_word_tokenize(text)
    elif lang == 'ko':
        return okt.morphs(text)
    elif lang == 'te':
        return indic_tokenize.trivial_tokenize(text)
    else:
        return text.split()

for lang in langForStat:
    num_train = df_train_clean[df_train_clean['lang'] == lang].shape[0]
    num_val = df_val_clean[df_val_clean['lang'] == lang].shape[0]
    numQuestions.append((lang, num_train, num_val))
    print(f"Language: {lang}, Train Questions: {num_train}, Validation Questions: {num_val}")

    df_train_lang = df_train_clean[df_train_clean['lang'] == lang].copy()
    df_val_lang = df_val_clean[df_val_clean['lang'] == lang].copy()

    df_train_lang['tokens'] = df_train_lang['question'].apply(lambda x: tokenize_by_lang(x, lang))
    df_val_lang['tokens'] = df_val_lang['question'].apply(lambda x: tokenize_by_lang(x, lang))

    df_train_lang['wordcount'] = df_train_lang['tokens'].apply(len)

    if not df_train_lang.empty:
        maxId = df_train_lang['wordcount'].idxmax()
        max_words = df_train_lang.loc[maxId, "wordcount"]
        print(f"  Word count: {max_words}")

    total_train_words = df_train_lang['wordcount'].sum()
    total_val_words = df_val_lang['tokens'].apply(len).sum()
    totalWordCount.append((lang, total_train_words, total_val_words))
    print(f"Language: {lang}, Train Total Words: {total_train_words}, Validation Total Words: {total_val_words}")

    all_words = [w for toks in df_train_lang['tokens'] for w in toks]
    word_counts = Counter(all_words)
    distinctWordCount.append((lang, len(word_counts)))
    print(f"Language: {lang}, Distinct Words: {len(word_counts)}")

    all_chars = [ch for w in word_counts.keys() for ch in w]
    char_counts = Counter(all_chars)
    distinctCharCount.append((lang, len(char_counts)))
    print(f"Language: {lang}, Distinct Characters: {len(char_counts)}")

    print(f"Language: {lang}, Calculated Total Words from WordDict: {sum(word_counts.values())}")

    top5Words = word_counts.most_common(5)
    print(f"Language: {lang}, Top 5 Words: {top5Words}")


Language: ar, Train Questions: 2558, Validation Questions: 415
  Word count: 16
Language: ar, Train Total Words: 16300, Validation Total Words: 2623
Language: ar, Distinct Words: 5406
Language: ar, Distinct Characters: 104
Language: ar, Calculated Total Words from WordDict: 16300
Language: ar, Top 5 Words: [('في', 593), ('من', 587), ('متى', 536), ('ما', 443), ('هو', 349)]
Language: ko, Train Questions: 2422, Validation Questions: 356
  Word count: 26
Language: ko, Train Total Words: 19813, Validation Total Words: 2893
Language: ko, Distinct Words: 3323
Language: ko, Distinct Characters: 819
Language: ko, Calculated Total Words from WordDict: 19813
Language: ko, Top 5 Words: [('는', 1094), ('인가', 1076), ('은', 1074), ('의', 747), ('무엇', 607)]
Language: te, Train Questions: 1355, Validation Questions: 384
  Word count: 14
Language: te, Train Total Words: 7737, Validation Total Words: 2306
Language: te, Distinct Words: 2415
Language: te, Distinct Characters: 91
Language: te, Calculated Total

In [ ]:
def arabicClassifier(question, context):
    goodWords = ['متى','ما','هو','هي','كم','عدد','أول','في']
    badWords = ['هل', 'يمكن']
    if any(word in question for word in badWords):
        return False
    if any(word in question for word in goodWords):
        return True
    else:
        return True

def koreanClassifier(question, context):
    goodWords = ['가장', '무엇인가', '언제', '몇']
    badWords = ['수 ']
    if any(word in question for word in badWords):
        return False
    if any(word in question for word in goodWords):
        return True
    else:
        return True

def teluguClassifier(question, context):
    goodWords = []
    badWords = ['విస్తీర్ణం', 'జనాభా', 'ఆఫ్రికాలో']
    if any(word in question for word in badWords):
        return False
    if any(word in question for word in goodWords):
            return True
    else:
        return True

arabicDf = df_val_clean[df_val_clean['lang'] == 'ar'].copy()
arabicDf['prediction'] = arabicDf.apply(lambda row: arabicClassifier(row['question'], row['context']), axis=1)
accuracy = (arabicDf['answerable'] == arabicDf['prediction']).mean()
print(f"Arabic Classifier Accuracy (validation): {accuracy * 100:.2f}%")
print(f"True distribution in validation set: {arabicDf['answerable'].value_counts(normalize=True).to_dict()}")

koreanDf = df_val_clean[df_val_clean['lang'] == 'ko'].copy()
koreanDf['prediction'] = koreanDf.apply(lambda row: koreanClassifier(row['question'], row['context']), axis=1)
accuracy = (koreanDf['answerable'] == koreanDf['prediction']).mean()
print(f"Korean Classifier Accuracy (validation): {accuracy * 100:.2f}%")
print(f"True distribution in validation set: {koreanDf['answerable'].value_counts(normalize=True).to_dict()}")

teluguDf = df_val_clean[df_val_clean['lang'] == 'te'].copy()
teluguDf['prediction'] = teluguDf.apply(lambda row: teluguClassifier(row['question'], row['context']), axis=1)
accuracy = (teluguDf['answerable'] == teluguDf['prediction']).mean()
print(f"Telugu Classifier Accuracy (validation): {accuracy * 100:.2f}%")
print(f"True distribution in validation set: {teluguDf['answerable'].value_counts(normalize=True).to_dict()}")

languages = ['ar', 'ko', 'te']

for lang in languages:
    train_subset = df_train_clean[df_train_clean['lang'] == lang]
    train_dist = train_subset['answerable'].value_counts(normalize=True).to_dict()

    val_subset = df_val_clean[df_val_clean['lang'] == lang]
    val_dist = val_subset['answerable'].value_counts(normalize=True).to_dict()

    print(f"\n=== {lang.upper()} ===")
    print(f"Training set distribution: {train_dist}")
    print(f"Validation set distribution: {val_dist}")

train_dist_all = df_train_clean['answerable'].value_counts(normalize=True).to_dict()
val_dist_all = df_val_clean['answerable'].value_counts(normalize=True).to_dict()

print("\n=== All languages combined ===")
print(f"Training set distribution: {train_dist_all}")
print(f"Validation set distribution: {val_dist_all}")


Arabic Classifier Accuracy (validation): 96.87%
True distribution in validation set: {True: 0.8746987951807229, False: 0.12530120481927712}
Korean Classifier Accuracy (validation): 94.94%
True distribution in validation set: {True: 0.9466292134831461, False: 0.05337078651685393}
Telugu Classifier Accuracy (validation): 79.17%
True distribution in validation set: {True: 0.7578125, False: 0.2421875}

=== AR ===
Training set distribution: {True: 0.900312744331509, False: 0.09968725566849101}
Validation set distribution: {True: 0.8746987951807229, False: 0.12530120481927712}

=== KO ===
Training set distribution: {True: 0.9739884393063584, False: 0.02601156069364162}
Validation set distribution: {True: 0.9466292134831461, False: 0.05337078651685393}

=== TE ===
Training set distribution: {True: 0.966789667896679, False: 0.033210332103321034}
Validation set distribution: {True: 0.7578125, False: 0.2421875}

=== All languages combined ===
Training set distribution: {True: 0.9099915270807535,

In [ ]:
for lang, df in [('Arabic', arabicDf), ('Korean', koreanDf), ('Telugu', teluguDf)]:
    print(f"\n{lang} classification report:")
    print(classification_report(df['answerable'], df['prediction']))


Arabic classification report:
              precision    recall  f1-score   support

       False       0.81      0.98      0.89        52
        True       1.00      0.97      0.98       363

    accuracy                           0.97       415
   macro avg       0.90      0.97      0.93       415
weighted avg       0.97      0.97      0.97       415


Korean classification report:
              precision    recall  f1-score   support

       False       0.57      0.21      0.31        19
        True       0.96      0.99      0.97       337

    accuracy                           0.95       356
   macro avg       0.76      0.60      0.64       356
weighted avg       0.94      0.95      0.94       356


Telugu classification report:
              precision    recall  f1-score   support

       False       0.72      0.23      0.34        93
        True       0.80      0.97      0.88       291

    accuracy                           0.79       384
   macro avg       0.76      0.60  

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def translate_text(text, src_lang, tgt_lang="en"):
    lang_map = {"ar": "ar_AR", "ko": "ko_KR", "te": "te_IN", "en": "en_XX", "da": "da_DK"}
    tokenizer.src_lang = lang_map[src_lang]

    encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    generated_tokens = model.generate(**encoded,forced_bos_token_id=tokenizer.lang_code_to_id[lang_map[tgt_lang]],max_length=128, num_beams=5)

    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

def translate_column(df, column_name, src_lang_col):
    tqdm.pandas(desc=f"Translating {column_name}")
    df[column_name + "_eng"] = df.progress_apply(lambda row: translate_text(row[column_name], src_lang=row[src_lang_col], tgt_lang="en"),axis=1)
    return df

output_dir = "/content/drive/MyDrive/translated_data"
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(output_dir, "df_train_translated.csv")
val_path = os.path.join(output_dir, "df_val_translated.csv")

if os.path.exists(train_path) and os.path.exists(val_path):
    print("Found existing translated files, loading...")
    df_train_trans = pd.read_csv(train_path)
    df_val_trans = pd.read_csv(val_path)
else:
    print("No translated files found, translating...")
    df_train_trans = df_train[df_train['lang'].isin(['ar', 'ko', 'te'])].reset_index(drop=True)
    df_val_trans = df_val[df_val['lang'].isin(['ar', 'ko', 'te'])].reset_index(drop=True)

    df_train_trans = translate_column(df_train_trans, column_name="question", src_lang_col="lang")
    df_val_trans = translate_column(df_val_trans, column_name="question", src_lang_col="lang")

    df_train_trans.to_csv(train_path, index=False, encoding="utf-8-sig")
    df_val_trans.to_csv(val_path, index=False, encoding="utf-8-sig")
    print(f"Translated data saved to:\n{train_path}\n{val_path}")

print(df_train_trans[['question', 'question_eng']].head())


Using device: cpu
Found existing translated files, loading...
                     question                                  question_eng
0           30년 전쟁의 승자는 누구인가?         Who is the winner of 30 years of war?
1             엑스선은 누가 발견하였는가?                         Who found the X-rays?
2  아테네에서 언제 가장 최근의 올림픽이 올렸나요?         When was the last Olympics in Athens?
3      세상에서 가장 오래된 방송사는 무엇인가?  What is the oldest broadcaster in the world?
4             팔레스타인 수도는 어딘가요?            Where is the capital of Palestine?


In [ ]:
def word_overlap(context, question):

    context_words = set(context.lower().split())
    question_words = set(question.lower().split())

    overlap = context_words.intersection(question_words)

    if len(question_words) == 0:
        return 0.0
    return len(overlap) / len(question_words)

In [ ]:
df_train_trans['word_overlap'] = df_train_trans.apply(lambda row: word_overlap(row['context'], row['question_eng']), axis=1)
df_val_trans['word_overlap'] = df_val_trans.apply(lambda row: word_overlap(row['context'], row['question_eng']), axis=1)

langs = {'ar': 'Arabic', 'ko': 'Korean', 'te': 'Telugu'}
thresholds = np.arange(0.15, 0.6, 0.05)

best_thresholds = {}

for code, name in langs.items():
    subset = df_val_trans[df_val_trans['lang'] == code]
    if subset.empty:
        print(f"\nNo samples for {name}")
        continue

    best_t, best_f1 = 0, 0
    for t in thresholds:
        preds = (subset['word_overlap'] > t).astype(int)
        f1 = f1_score(subset['answerable'], preds)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    best_thresholds[code] = best_t
    print(f"{name} ({code}) → Best threshold: {best_t:.2f} | F1: {best_f1:.4f}")

for code, name in langs.items():
    subset = df_val_trans[df_val_trans['lang'] == code]
    if subset.empty:
        continue

    t = best_thresholds[code]
    subset['pred_answerable'] = (subset['word_overlap'] > t).astype(int)

    print(f"\n=== {name} ({code}) — threshold {t:.2f} ===")
    print(classification_report(subset['answerable'], subset['pred_answerable'],
                                target_names=['Impossible', 'Answerable'],
                                digits=3))


Arabic (ar) → Best threshold: 0.15 | F1: 0.8949
Korean (ko) → Best threshold: 0.15 | F1: 0.9240
Telugu (te) → Best threshold: 0.15 | F1: 0.8141

=== Arabic (ar) — threshold 0.15 ===
              precision    recall  f1-score   support

  Impossible      0.139     0.096     0.114        52
  Answerable      0.876     0.915     0.895       363

    accuracy                          0.812       415
   macro avg      0.507     0.505     0.504       415
weighted avg      0.784     0.812     0.797       415


=== Korean (ko) — threshold 0.15 ===
              precision    recall  f1-score   support

  Impossible      0.057     0.105     0.074        19
  Answerable      0.947     0.902     0.924       337

    accuracy                          0.860       356
   macro avg      0.502     0.504     0.499       356
weighted avg      0.900     0.860     0.879       356


=== Telugu (te) — threshold 0.15 ===
              precision    recall  f1-score   support

  Impossible      0.275     0.151

/tmp/ipython-input-3775885282.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['pred_answerable'] = (subset['word_overlap'] > t).astype(int)
/tmp/ipython-input-3775885282.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['pred_answerable'] = (subset['word_overlap'] > t).astype(int)
/tmp/ipython-input-3775885282.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the

# Week 41

In [ ]:
test_path = os.path.join(output_dir, "test.json")
df_test = pd.read_json(test_path)

df_test_trans = translate_column(df_test, column_name="question", src_lang_col="lang")
df_test_trans['word_overlap'] = df_test_trans.apply(lambda row: word_overlap(row['context'], row['question_eng']), axis=1)

threshold = 0.15

df_test_trans["pred_answerable"] = (df_test_trans["word_overlap"] > threshold).astype(int)

print(f"Evaluating on test set with threshold = {threshold}")
print(classification_report(
    df_test_trans["answerable"],
    df_test_trans["pred_answerable"],
    target_names=["Unanswerable", "Answerable"],
    digits=3
))

Translating question: 100%|██████████| 20/20 [01:55<00:00,  5.75s/it]

Evaluating on test set with threshold = 0.15
              precision    recall  f1-score   support

Unanswerable      0.400     0.400     0.400         5
  Answerable      0.800     0.800     0.800        15

    accuracy                          0.700        20
   macro avg      0.600     0.600     0.600        20
weighted avg      0.700     0.700     0.700        20

